# Phase 2: Data Cleaning

**Goal:** Fix all inconsistencies so downstream feature engineering works cleanly.

**What we fix:**
1. Team name standardization (19 names → 12 clean names)
2. Season format (mixed '2007/08' and 2023 → clean year)
3. Stage / playoff flag
4. Nulls — wicket_kind and player_out nulls are EXPECTED (most balls aren't wickets)
5. Extract clean winner column from match_won_by

**Output:** `data/processed/cleaned_balls.csv` and `data/processed/cleaned_matches.csv`

## 1. Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# low_memory=False silences the mixed-type warning we saw in Phase 1
ball_df = pd.read_csv('../data/raw/matches.csv', parse_dates=['date'], low_memory=False)

print(f'Loaded: {len(ball_df):,} rows, {ball_df.shape[1]} columns')

## 2. Team Name Standardization

**Problem:** The dataset has 19 different team name strings, but IPL has only had ~12 franchises.
Some teams were renamed, some folded, some have typos.

Examples:
- `'Delhi Daredevils'` → `'Delhi Capitals'` (renamed in 2019)
- `'Kings XI Punjab'` → `'Punjab Kings'` (renamed in 2021)
- `'Royal Challengers Bangalore'` → `'Royal Challengers Bengaluru'` (renamed in 2024)
- `'Rising Pune Supergiant'` → `'Rising Pune Supergiants'` (typo in one season)
- Defunct teams (Deccan Chargers, Kochi Tuskers, etc.) keep their original name — they existed and we need their history

In [ ]:
# Team name mapping: old/variant name → canonical current name
# Key rule: if a team is STILL ACTIVE today, map old names to the current one
# Defunct teams stay as-is (their historical records are valid)
TEAM_NAME_MAP = {
    # Renamed teams — map to current name
    'Delhi Daredevils'            : 'Delhi Capitals',
    'Kings XI Punjab'             : 'Punjab Kings',
    'Royal Challengers Bangalore' : 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant'      : 'Rising Pune Supergiants',   # typo fix
}
# Teams NOT in this map keep their name as-is
# (CSK, MI, KKR, RR, SRH, GT, LSG, Deccan Chargers, Pune Warriors, etc.)

# Apply the mapping to all team columns
# .replace() swaps values according to the dictionary — values not in dict are unchanged
for col in ['batting_team', 'bowling_team', 'toss_winner', 'match_won_by']:
    if col in ball_df.columns:
        ball_df[col] = ball_df[col].replace(TEAM_NAME_MAP)

# Verify: check the unique team names after mapping
all_teams_after = sorted(pd.concat([ball_df['batting_team'], ball_df['bowling_team']]).unique())
print(f'Team names after cleaning ({len(all_teams_after)} unique):')
for t in all_teams_after:
    print(f'  {t}')

# Extract the correct year from season string
# Problem: '2007/08' → our old logic gave 2007, but IPL season played in 2008
# Fix: for cross-year seasons (YYYY/YY format), take the SECOND year (the one matches were played in)
# '2007/08' → 2008, '2009/10' → 2010, '2020/21' → 2021
# Single-year seasons ('2023') stay as-is

def parse_season_year(season_str):
    season_str = str(season_str)
    if '/' in season_str:
        # e.g. '2007/08' → base year 2007, suffix '08' → 2008
        base_year = int(season_str.split('/')[0])
        suffix    = int(season_str.split('/')[1])
        # suffix is a 2-digit year — combine with century from base year
        return (base_year // 100) * 100 + suffix
    else:
        return int(season_str)

ball_df['season_year'] = ball_df['season'].apply(parse_season_year)

print('Season → season_year mapping:')
check = ball_df[['season','season_year']].drop_duplicates().sort_values('season_year')
print(check.to_string(index=False))

In [ ]:
# Extract start year from season string
# '2007/08' → take everything before '/' → '2007' → int 2007
# '2023'    → no '/', take as-is → int 2023
ball_df['season_year'] = (
    ball_df['season']
    .astype(str)              # ensure it's a string first
    .str.split('/')           # split on '/' → ['2007', '08'] or ['2023']
    .str[0]                   # take the first element → '2007' or '2023'
    .astype(int)              # convert to integer
)

print('Season year values:')
print(sorted(ball_df['season_year'].unique()))

## 4. Playoff Flag

The `stage` column tells us if a match is a playoff or group stage.
We'll create a simple `is_playoff` binary column — useful as a feature later.

In [ ]:
# Print unique stage values to understand what we're working with
print('Stage values:')
print(ball_df['stage'].value_counts())

In [ ]:
# Create is_playoff: 1 if it's any kind of knockout match, 0 if group stage or unknown
PLAYOFF_STAGES = {
    'Semi Final', 'Final', 'Qualifier 1', 'Qualifier 2',
    'Eliminator', 'Elimination Final', '3rd Place Play-Off'
}

ball_df['is_playoff'] = ball_df['stage'].isin(PLAYOFF_STAGES).astype(int)

print(f'Playoff deliveries : {ball_df["is_playoff"].sum():,}')
print(f'Group stage        : {(ball_df["is_playoff"] == 0).sum():,}')

## 5. Null Handling

From Phase 1 we saw only two columns have nulls:
- `wicket_kind` — null on 264,382 rows
- `player_out`  — null on 264,382 rows

**These nulls are CORRECT.** Most deliveries are not wickets — the wicket columns are only populated when a wicket falls. We don't drop these rows; we fill the nulls with a placeholder.

In [ ]:
# Fill wicket nulls with 'none' — means no wicket on that ball
# fillna() replaces NaN with the value you provide
ball_df['wicket_kind'] = ball_df['wicket_kind'].fillna('none')
ball_df['player_out']  = ball_df['player_out'].fillna('none')

# Add a binary is_wicket column — cleaner than checking wicket_kind != 'none'
ball_df['is_wicket'] = (ball_df['wicket_kind'] != 'none').astype(int)

print(f'Total wickets in dataset : {ball_df["is_wicket"].sum():,}')

# Verify no more nulls in these columns
print(f'Nulls in wicket_kind     : {ball_df["wicket_kind"].isnull().sum()}')
print(f'Nulls in player_out      : {ball_df["player_out"].isnull().sum()}')

## 6. Build Clean Matches Table

Deduplicate ball-by-ball into one row per match with all match-level fields.
Also extract the winner properly — `match_won_by` already contains the winning team name.

In [ ]:
# Columns we want in the match-level table
MATCH_COLS = [
    'match_id', 'date', 'season_year', 'venue', 'city',
    'toss_winner', 'toss_decision',
    'match_won_by',   # the winning team name
    'win_outcome',    # e.g. '6 wickets' or '45 runs'
    'stage', 'is_playoff'
]

available_cols = [c for c in MATCH_COLS if c in ball_df.columns]

matches_clean = (
    ball_df[available_cols]
    .drop_duplicates(subset='match_id')
    .reset_index(drop=True)
)

# Rename for clarity
matches_clean = matches_clean.rename(columns={'match_won_by': 'winner'})

print(f'Clean matches: {len(matches_clean):,}')
matches_clean.head()

In [ ]:
# Integrity check: does every match have a winner?
no_winner = matches_clean['winner'].isnull().sum()
print(f'Matches with no winner: {no_winner}')

# Check what they look like (could be no-result / abandoned matches)
if no_winner > 0:
    print(matches_clean[matches_clean['winner'].isnull()][['match_id','date','stage','win_outcome']].head(10))

In [ ]:
# Drop no-result matches (abandoned, no winner) — can't use these for training
before = len(matches_clean)
matches_clean = matches_clean.dropna(subset=['winner']).reset_index(drop=True)
after = len(matches_clean)
print(f'Dropped {before - after} no-result matches')
print(f'Clean matches remaining: {after:,}')

In [ ]:
# Final check: are all winners valid team names?
all_teams = set(pd.concat([ball_df['batting_team'], ball_df['bowling_team']]).unique())
invalid_winners = matches_clean[~matches_clean['winner'].isin(all_teams)]['winner'].unique()
print(f'Winners not in team list: {invalid_winners}')
# Should be empty — if not, we have more name cleaning to do

## 7. Save Cleaned Data

In [ ]:
# Save cleaned ball-by-ball data
ball_df.to_csv('../data/processed/cleaned_balls.csv', index=False)
print(f'Saved cleaned_balls.csv  — {len(ball_df):,} rows')

# Save cleaned match-level data
matches_clean.to_csv('../data/processed/cleaned_matches.csv', index=False)
print(f'Saved cleaned_matches.csv — {len(matches_clean):,} rows')

## 8. Cleaning Summary

In [ ]:
print('=' * 50)
print('CLEANING SUMMARY')
print('=' * 50)
print(f'Team names standardized : Delhi Daredevils → Delhi Capitals')
print(f'                          Kings XI Punjab  → Punjab Kings')
print(f'                          RCB Bangalore    → RCB Bengaluru')
print(f'                          Rising Pune typo fixed')
print(f'Season format           : mixed → clean integer year')
print(f'Wicket nulls            : filled with "none" + is_wicket column added')
print(f'Playoff flag            : is_playoff column added')
print(f'Final match count       : {len(matches_clean):,} (after dropping no-results)')
print(f'Total nulls remaining   : {ball_df.isnull().sum().sum()}')
print('=' * 50)

## Your Tasks (Before Phase 3)

```
□ Manually verify 3 name mappings:
  - Find a match from 2018 with 'Delhi Daredevils' in the raw file
  - Confirm cleaned_balls.csv shows 'Delhi Capitals' for the same match_id
  - Do the same for 'Kings XI Punjab' and 'Royal Challengers Bangalore'

□ For each of these, write ONE sentence explaining what it does:
  - df.replace(TEAM_NAME_MAP)
  - str.split('/').str[0]
  - df.isin(PLAYOFF_STAGES)
  - df.fillna('none')
  - df.dropna(subset=['winner'])

□ Answer: why didn't we drop rows where wicket_kind is null?

□ Add to interview notebook:
  Q: "How did you handle missing data?"
  A: [your answer — distinguish between expected nulls vs real missing data]

  Q: "Why not just drop all rows with any null?"
  A: [your answer]
```